# T1 · Fact, Dimension, Degenerate or Drop?
**Model context:** one row of `FactShipment` = one shipment leg (transaction fact). Existing measures: `WeightKG`, `CostPKR`. Existing keys: City, Route, Mode, Status, Date + degenerate `ShipmentID`.

Verdict key: **F** fact · **D** dimension attribute · **DD** degenerate dimension · **X** neither.

## 1 · The twelve verdicts

In [1]:
import pandas as pd
pd.set_option("display.max_colwidth", None, "display.width", 250)

rows = [
 (1,  "Driver_Name",            "D",  "DimDriver",
      "Descriptive text about a person; stored once per driver, fact carries DriverKey (-1 where no driver is recorded)."),
 (2,  "Days_Late",              "F",  "FactShipment",
      "Additive count of days per leg, summable across route/mode/month; NULL (not 0) while In Transit, with DeliveredLegCount as the denominator for averages."),
 (3,  "Invoice_Number",         "DD", "FactShipment",
      "A transaction identifier with no attributes of its own; it groups legs, so it sits in the fact as a key-like column with no dimension table."),
 (4,  "Discount_Percent",       "F",  "FactShipment (as Discount_PKR + GrossCostPKR)",
      "TRAP: a ratio is non-additive and averaging it gives the wrong answer; store the numerator and denominator, compute % as DIVIDE(SUM, SUM)."),
 (5,  "Truck_Registration_No",  "D",  "DimTruck",
      "TRAP: looks like a degenerate key but a truck recurs across many legs and carries attributes (make, capacity, owner), so it is a dimension; DD is only for identifiers with nothing to say about themselves."),
 (6,  "Container_Count",        "F",  "FactShipment",
      "Additive count of containers on the leg, valid across every dimension; store 0 (not NULL) for non-containerised rail/road legs."),
 (7,  "Customer_Credit_Rating", "D",  "DimCustomer (SCD Type 2)",
      "A property of the customer, not of the shipment event; it changes over time, so Type 2 keeps the rating that applied at shipment time."),
 (8,  "Cost_Per_Tonne_KM",      "F",  "FactShipment (as CostPKR + TonneKM)",
      "TRAP: a ratio; store CostPKR and TonneKM (tonnes x route km, additive) and divide the two sums at report time, so it is right at every level."),
 (9,  "Row_Inserted_Timestamp", "X",  "ETL audit log / staging (optionally DimAudit via AuditKey)",
      "TRAP: load metadata, not business context; it describes the pipeline, not the shipment, and a load time in the fact would be mistaken for ShipDate."),
 (10, "Fuel_Surcharge_PKR",     "F",  "FactShipment",
      "Additive money amount at leg grain; keep it separate from CostPKR (never inside it) so the existing 4,808,000 total does not move."),
 (11, "Weather_Condition",      "D",  "DimWeather (small dimension)",
      "Descriptive text with low cardinality (Clear, Rain, Fog, Flood); stored once, fact carries WeatherKey."),
 (12, "Is_Hazardous_Cargo",     "D",  "DimShipmentFlags (junk dimension)",
      "A text-labelled flag ('Hazardous'/'Non-hazardous'); flags belong in a small junk dimension, not as text in the fact."),
]
t1 = pd.DataFrame(rows, columns=["#", "Column", "Verdict", "Lives in", "Justification"])
t1["Tricky"] = t1["#"].isin([4, 5, 8, 9]).map({True: "yes", False: ""})
t1

,#,Column,Verdict,Lives in,Justification,Tricky
0,1,Driver_Name,D,DimDriver,"Descriptive text about a person; stored once per driver, fact carries DriverKey (-1 where no driver is recorded).",
1,2,Days_Late,F,FactShipment,"Additive count of days per leg, summable across route/mode/month; NULL (not 0) while In Transit, with DeliveredLegCount as the denominator for averages.",
2,3,Invoice_Number,DD,FactShipment,"A transaction identifier with no attributes of its own; it groups legs, so it sits in the fact as a key-like column with no dimension table.",
3,4,Discount_Percent,F,FactShipment (as Discount_PKR + GrossCostPKR),"TRAP: a ratio is non-additive and averaging it gives the wrong answer; store the numerator and denominator, compute % as DIVIDE(SUM, SUM).",yes
4,5,Truck_Registration_No,D,DimTruck,"TRAP: looks like a degenerate key but a truck recurs across many legs and carries attributes (make, capacity, owner), so it is a dimension; DD is only for identifiers with nothing to say about themselves.",yes
5,6,Container_Count,F,FactShipment,"Additive count of containers on the leg, valid across every dimension; store 0 (not NULL) for non-containerised rail/road legs.",
6,7,Customer_Credit_Rating,D,DimCustomer (SCD Type 2),"A property of the customer, not of the shipment event; it changes over time, so Type 2 keeps the rating that applied at shipment time.",
7,8,Cost_Per_Tonne_KM,F,FactShipment (as CostPKR + TonneKM),"TRAP: a ratio; store CostPKR and TonneKM (tonnes x route km, additive) and divide the two sums at report time, so it is right at every level.",yes
8,9,Row_Inserted_Timestamp,X,ETL audit log / staging (optionally DimAudit via AuditKey),"TRAP: load metadata, not business context; it describes the pipeline, not the shipment, and a load time in the fact would be mistaken for ShipDate.",yes
9,10,Fuel_Surcharge_PKR,F,FactShipment,"Additive money amount at leg grain; keep it separate from CostPKR (never inside it) so the existing 4,808,000 total does not move.",


## 2 · Rule check (design must obey the six rules)

In [2]:
# Proposed FactShipment columns after T1: keys, degenerate columns, additive measures only
fact_keys   = ["DateKey", "OriginCityKey", "DestCityKey", "RouteKey", "ModeKey", "StatusKey",
               "DriverKey", "TruckKey", "CustomerKey", "WeatherKey", "FlagKey", "AuditKey"]
fact_dd     = ["ShipmentID", "Invoice_Number"]
fact_meas   = ["WeightKG", "CostPKR", "Days_Late", "DeliveredLegCount", "Container_Count",
               "Fuel_Surcharge_PKR", "Discount_PKR", "GrossCostPKR", "TonneKM", "ShipmentCount"]

banned_ratio_cols = {"Discount_Percent", "Cost_Per_Tonne_KM"}
descriptive_cols  = set(t1.loc[t1.Verdict.isin(["D", "X"]), "Column"])

fact_cols = set(fact_keys + fact_dd + fact_meas)
assert not (fact_cols & banned_ratio_cols),  "ratio stored in fact"
assert not (fact_cols & descriptive_cols),   "descriptive/audit column in fact"
assert all(k.endswith("Key") for k in fact_keys), "non-integer-key style column"
assert t1.Verdict.isin(["F", "D", "DD", "X"]).all() and len(t1) == 12
print("All rule checks pass")
print(t1.Verdict.value_counts().to_dict())

All rule checks pass
{'D': 5, 'F': 5, 'DD': 1, 'X': 1}


**Every dimension gets a `-1` Unknown row:** `DimDriver`, `DimTruck`, `DimCustomer`, `DimWeather`, `DimShipmentFlags`, `DimAudit` each carry key `-1`, so marine/rail legs with no driver or truck land in a countable row.

## 3 · Totals baseline (must not move)

In [3]:
import os
p = next(x for x in ["Shipments_Unified.csv", "../Shipments_Unified.csv", "/mnt/user-data/uploads/Shipments_Unified.csv"] if os.path.exists(x))
src = pd.read_csv(p)
assert len(src) == 34 and src.WeightKG.sum() == 578_650 and src.CostPKR.sum() == 4_808_000
print(f"{len(src)} rows | {src.WeightKG.sum():,} kg | PKR {src.CostPKR.sum():,}  -> baseline holds; new measures are added as separate columns")

34 rows | 578,650 kg | PKR 4,808,000  -> baseline holds; new measures are added as separate columns


## 4 · The three defence questions

**Q1. Which of your X answers would an analyst fight you over, and what is their best argument?**
`Row_Inserted_Timestamp`. Their best argument: lineage and reproducibility. When a late-arriving fact is loaded, they need to answer "what did this report show on 3 March?" and to trace a wrong number to its load batch, and only the insert time tells them. That argument is valid, but it justifies an `AuditKey` (integer) pointing to a `DimAudit` row (batch id, load time, source file), not a timestamp column in the fact. The need is met and the fact stays free of descriptive columns.

**Q2. Name one column marked D that deserves its own dimension table, and three attributes it would carry.**
`Driver_Name` → `DimDriver`, carrying `LicenceCategory`, `EmploymentType` (employee/contract) and `HomeDepotCityKey`. A name alone is one attribute; once those attributes exist, repeating them on every leg would store them once per fact row instead of once per driver.

**Q3. If Days_Late is a fact, which dimension supplies the deadline it is measured against?**
`DimDate`, used as a role-playing dimension: the fact carries `PromisedDeliveryDateKey` and `ActualDeliveryDateKey`, both pointing to `DimDate`, and `Days_Late` is the difference. The promised date is computed at load from `ShipDate` plus the route's standard transit days (a `DimRoute` attribute) and frozen in the fact, so changing a route's SLA later does not rewrite history.